-  <span style="color: rgb(32, 33, 34); font-family: Lato, &quot;Lucida Sans Unicode&quot;, &quot;Lucida Grande&quot;, sans-serif; font-size: 40px; letter-spacing: 0.2px; background-color: yellow;"><b><u>HW 6 - Chapter 6: Set operators</u></b></span>

### <mark>**1\. Products listed in both Shopping Carts and Special Offers**</mark>

**Proposition:**  
Find products that are both in shopping carts and linked to special offers.

**Explanation:**  
Uses `INTERSECT` to return common products between two different promotional pipelines.

**Why it’s special:**  
These products are likely top-selling and suitable for upselling or bundling strategies.

In [ ]:
USE AdventureWorks2019;
SELECT ProductID FROM Sales.ShoppingCartItem
INTERSECT
SELECT ProductID FROM Sales.SpecialOfferProduct;


### **<mark>2\. Employees who are _not_ in Sales</mark>**

<mark>**Proposition****:**</mark>  
Return a list of employees who are not working in the sales department.

**Explanation:**

This uses `EXCEPT` to remove all salespeople from the general employee pool.

**Why it’s special:**  
Great for isolating technical or administrative staff for internal communication or HR training sessions.

In [ ]:
USE AdventureWorks2019;
SELECT BusinessEntityID FROM HumanResources.Employee
EXCEPT
SELECT BusinessEntityID FROM Sales.SalesPerson;


### **<mark>3\. Products that have never been reviewed</mark>**

**Proposition:**

List products that are sold but have no customer reviews.  

**Explanation:**

Uses `EXCEPT` to show unreviewed products, which may need promotion or feedback gathering.

**Why it’s special:**  
Helps businesses identify products needing attention or customer education.

List products that are sold but have no customer reviews.

In [ ]:
USE AdventureWorks2019;
SELECT ProductID FROM Production.Product
EXCEPT
SELECT ProductID FROM Production.ProductReview;


### **<mark>4\. Employees who worked multiple shifts (based on department history)</mark>**

**Proposition:**  
List employees who have been assigned to more than one shift.**Explanation:**

Combines department history and pay change data to isolate multi-role employees.

**Why it’s special:**  
These employees may be adaptable and strong candidates for promotion.

In [ ]:
USE AdventureWorks2019;
SELECT BusinessEntityID FROM HumanResources.EmployeeDepartmentHistory
INTERSECT
SELECT BusinessEntityID FROM HumanResources.EmployeePayHistory;


### **<mark>5 All Business Entities who are either Employees or Job Candidates (Using `UNION`)</mark>**

**Proposition:**  
Show BusinessEntityIDs that exist in either the Employee or Job Candidate list.

**Explanation:**  
`UNION` removes duplicates, showing all individuals involved in the company, either currently working or being considered.

**Why it’s special:**  
This helps HR maintain a complete view of internal and external talent pools.

In [ ]:
USE AdventureWorks2019;
SELECT BusinessEntityID FROM HumanResources.Employee
UNION
SELECT BusinessEntityID FROM HumanResources.JobCandidate;


### **<mark>6\. All Shipped Orders and Orders Pending Shipment (Using `UNION ALL`)</mark>**

**Proposition:**  
List all orders—shipped and not yet shipped—with their status, even if some overlap.

**Explanation:**

`UNION ALL` keeps all results, even if the same order ID shows up in both (e.g., partial shipments).

**Why it’s special:**  
Good for shipping reports or dashboards where you need _all_ statuses shown clearly.

In [ ]:
USE AdventureWorks2019;
SELECT SalesOrderID, 'Shipped' AS Status FROM Sales.SalesOrderHeader WHERE ShipDate IS NOT NULL
UNION ALL
SELECT SalesOrderID, 'Pending' AS Status FROM Sales.SalesOrderHeader WHERE ShipDate IS NULL;


### **<mark>7\. Phone Numbers that are used by multiple people or shared types (Using `UNION ALL` with `INTERSECT`)</mark>**

**<mark>Proposition:</mark>**  
Combine:

- Phone numbers assigned to more than one person
- Phone types used by at least two people
- **Explanation:**
- `UNION ALL` <span style="color: var(--vscode-foreground);">here joins two different dimensions of “shared” phone data: numbers and types.</span>
- **Why it’s special:**
- Helps detect shared contact details—useful for fraud prevention, auditing, or family grouping.

In [ ]:
USE AdventureWorks2019;
-- Shared numbers
SELECT PhoneNumber FROM Person.PersonPhone
GROUP BY PhoneNumber
HAVING COUNT(BusinessEntityID) > 1

UNION ALL

-- Shared types
SELECT CAST(PhoneNumberTypeID AS VARCHAR) FROM Person.PersonPhone
GROUP BY PhoneNumberTypeID
HAVING COUNT(BusinessEntityID) > 1;


### **<mark>8\. All Territories That Are Either Current, Historical, or Both (Using `UNION` and `INTERSECT`)</mark>**

**Proposition:**  
Return a full list of all territory IDs that have ever existed, showing which ones were both current and historical.

**Explanation:**

Two queries: one for all-time history (with `UNION`) and one for overlap (with `INTERSECT`).

**Why it’s special:**  
Useful for regional performance comparisons or identifying long-standing operational zones.

In [ ]:
USE AdventureWorks2019;
-- All territories ever recorded
SELECT TerritoryID FROM Sales.SalesTerritory
UNION
SELECT TerritoryID FROM Sales.SalesTerritoryHistory;

-- Territories that are both current and historical
SELECT TerritoryID FROM Sales.SalesTerritory
INTERSECT
SELECT TerritoryID FROM Sales.SalesTerritoryHistory;


### **<mark>9\. All People Who Are Employees, Contacts, or Both (Using `UNION`, `INTERSECT`, `EXCEPT`)</mark>**

**Proposition:**  
List all people who are:

- Only employees
    
- Only contacts
    
- Both (overlap)
    
- **Explanation:**
    
    I use a mix of all three operators to categorize people by their involvement.
    
    **Why it’s special:**  
    Creates a holistic people profile system, good for CRM and HR analytics.

In [ ]:
USE AdventureWorks2019;
-- Only employees
SELECT BusinessEntityID FROM HumanResources.Employee
EXCEPT
SELECT BusinessEntityID FROM Person.BusinessEntityContact

UNION

-- Only contacts
SELECT BusinessEntityID FROM Person.BusinessEntityContact
EXCEPT
SELECT BusinessEntityID FROM HumanResources.Employee

UNION

-- Both
SELECT BusinessEntityID FROM HumanResources.Employee
INTERSECT
SELECT BusinessEntityID FROM Person.BusinessEntityContact;


### **<mark>10\. Customers Who Bought from Multiple Territories or None (Using `UNION`, `EXCEPT`</mark>)**

**Proposition:**  
Find:

- Customers who bought from multiple sales territories
    
- Customers who never had a territory assigned
    
- **Explanation:**  
    This query identifies unusually mobile or unassigned customers.
    
    **Why it’s special:**  
    Helps with sales tracking and adjusting territory boundaries.

In [ ]:
USE AdventureWorks2019;
-- Customers with orders in multiple territories
SELECT CustomerID FROM Sales.SalesOrderHeader
GROUP BY CustomerID
HAVING COUNT(DISTINCT TerritoryID) > 1

UNION

-- Customers with no territory assigned
SELECT CustomerID FROM Sales.Customer
EXCEPT
SELECT CustomerID FROM Sales.SalesOrderHeader WHERE TerritoryID IS NOT NULL;
